In [ ]:
import os
import cv2
import easyocr
import numpy as np
import re

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
def Webcam(cameras : int):
    if type(camera) == int and cameras:
        return cv2.VideoCapture(cameras)
    
camera = Webcam(0)

if not camera.isOpened():
    raise RuntimeError("Failed to Access Webcam")

reader = easyocr.Reader(
    ["en"],
    gpu=False
)

OCR_INTERVAL = 6
CONFIDENCE = 0.35

frame_count = 0
detections = []

def classify_text(text):

    cleaned = text.strip().replace(",", "").replace(" ", "")

    if re.fullmatch(
        r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)",
        cleaned
    ):
        return "Number"

    return "Text"


while True:

    success, frame = camera.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    frame = cv2.resize(
        frame,
        (960, 540)
    )

    frame_count += 1

    # Run OCR periodically to reduce processing load
    if frame_count % OCR_INTERVAL == 0:

        detections = reader.readtext(
            frame,
            detail=1,
            paragraph=False
        )

    for detection in detections:

        bounding_box, recognized_text, confidence = detection

        if confidence < CONFIDENCE:
            continue

        result_type = classify_text(
            recognized_text
        )

        if result_type == "Number":

            label = f"Number = {recognized_text}"
            color = (0, 255, 0)

        else:

            label = f"Text = {recognized_text}"
            color = (255, 180, 0)

        points = np.array(
            bounding_box,
            dtype=np.int32
        )

        x, y = points.min(axis=0)

        cv2.polylines(
            frame,
            [points],
            True,
            color,
            2
        )

        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.55
        thickness = 2

        max_width = frame.shape[1] - 10

        while (
            cv2.getTextSize(
                label,
                font,
                font_scale,
                thickness
            )[0][0] > max_width - 10
            and font_scale > 0.3
        ):
            font_scale -= 0.05

        text_size = cv2.getTextSize(
            label,
            font,
            font_scale,
            thickness
        )[0]

        text_width, text_height = text_size

        label_x = max(
            5,
            min(x, frame.shape[1] - text_width - 10)
        )

        label_y = max(
            text_height + 12,
            y - 10
        )

        cv2.rectangle(
            frame,
            (
                label_x,
                label_y - text_height - 8
            ),
            (
                label_x + text_width + 8,
                label_y + 5
            ),
            color,
            -1
        )

        cv2.putText(
            frame,
            label,
            (label_x + 4, label_y),
            font,
            font_scale,
            (0, 0, 0),
            thickness
        )

    cv2.imshow(
        "Text & Number Detector",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()